In [1]:
import numpy as np
from collections import defaultdict

In [2]:
N = 4                      # grid size
NUM_CELLS = N*N
NUM_STATES = 2**NUM_CELLS

gamma = 0.9
theta = 1e-6

In [3]:
def int_to_state(x):
    bits = [0]*NUM_CELLS
    for i in range(NUM_CELLS-1, -1, -1):
        bits[i] = x % 2
        x //= 2
    return tuple(bits)

In [4]:
states = [int_to_state(i) for i in range(NUM_STATES)]
goal_state = tuple([0]*NUM_CELLS)
actions = list(range(NUM_CELLS))

In [5]:
def toggle(state, action):
    new = list(state)
    i = action // N
    j = action % N

    def flip(r,c):
        idx = r*N + c
        new[idx] ^= 1

    flip(i,j)
    if i>0: flip(i-1,j)
    if i<N-1: flip(i+1,j)
    if j>0: flip(i,j-1)
    if j<N-1: flip(i,j+1)

    return tuple(new)

In [6]:
MDP = {s:{} for s in states}

for s in states:
    for a in actions:
        if s == goal_state:
            MDP[s][a] = [(1, s, 0, True)]
        else:
            ns = toggle(s,a)
            r = 1 if ns==goal_state else -1
            done = (ns==goal_state)
            MDP[s][a] = [(1, ns, r, done)]

In [7]:
V = {s:0.0 for s in states}
V[goal_state] = 0.0

while True:
    delta = 0
    Vnew = V.copy()

    for s in states:
        if s==goal_state: continue
        best = -1e18
        for a in actions:
            p,ns,r,_ = MDP[s][a][0]
            val = p*(r + gamma*V[ns])
            best = max(best,val)
        Vnew[s]=best
        delta = max(delta, abs(V[s]-best))

    V = Vnew
    if delta < theta:
        break

print("Value Iteration Converged")

Value Iteration Converged


In [8]:
policy = {}

for s in states:
    if s==goal_state:
        policy[s]=None
        continue
    best = -1e18
    besta = 0
    for a in actions:
        p,ns,r,_ = MDP[s][a][0]
        val = p*(r + gamma*V[ns])
        if val>best:
            best=val
            besta=a
    policy[s]=besta

print("Policy Extracted")

Policy Extracted


In [9]:
def print_board(state):
    for i in range(0,NUM_CELLS,N):
        for j in range(i,i+N):
            print(state[j], end=" ")
        print()

In [10]:
start_state = (0,0,0,1,
               0,1,1,0,
               0,1,0,1,
               0,1,0,0)

s = start_state
moves = 0
LIMIT = 100

print("Initial Board:")
print_board(s)
print()

while s!=goal_state and moves<LIMIT:
    a = policy[s]
    i = a//N
    j = a%N
    print("Move:", (i+1, j+1))
    s = toggle(s,a)
    print_board(s)
    print()
    moves+=1

if s==goal_state:
    print("DONE")
else:
    print("IMPOSSIBLE")

print("Moves Taken:", moves)

Initial Board:
0 0 0 1 
0 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
1 1 0 1 
1 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
0 0 0 1 
0 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
1 1 0 1 
1 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
0 0 0 1 
0 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
1 1 0 1 
1 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
0 0 0 1 
0 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
1 1 0 1 
1 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
0 0 0 1 
0 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
1 1 0 1 
1 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
0 0 0 1 
0 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
1 1 0 1 
1 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
0 0 0 1 
0 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
1 1 0 1 
1 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
0 0 0 1 
0 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
1 1 0 1 
1 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
0 0 0 1 
0 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
1 1 0 1 
1 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
0 0 0 1 
0 1 1 0 
0 1 0 1 
0 1 0 0 

Move: (1, 1)
1 1 0 1 
1 1 1 0 
0 1 0 1 
0 1 0 0 

In [11]:
test = tuple([1]*NUM_CELLS)
s=test
steps=0
while s!=goal_state and steps<100:
    s=toggle(s,policy[s])
    steps+=1

print("Steps from all-ones board:",steps)

Steps from all-ones board: 4
